# Theme

This project simulates a health insurance claim validation system that checks policy rules, patient eligibility, waiting periods, hospital conditions, and treatment costs, then decides whether a claim is approved, partially approved, or rejected—just like an insurance officer.

# Install

gradio → Creates the web interface where users enter patient and claim details.

python-dateutil → Helps calculate waiting periods and date differences accurately.

In [ ]:
!pip install gradio python-dateutil -q

Imports Cell → Loads libraries required for date calculations and building the Gradio interface.

In [ ]:
import gradio as gr
from datetime import datetime
from dateutil.relativedelta import relativedelta

here we can see some diseases ans their insurance conditions

POLICY_RULES Dictionary Cell → Contains predefined insurance rules for diseases like Heart Surgery, Diabetes, and Cancer Chemo.

parse_date() Function Cell → Converts user-entered date text into a valid date object for comparison.

evaluate_claim() Function Cell → Validates the claim by checking age limits, waiting periods, hospital type, and coverage limit, then decides approval or rejection.

Age Check Block → Ensures the patient falls within the allowed age range for the selected treatment.

Policy Date Validation Block → Confirms the policy start and claim date format are correct.

Waiting Period Block → Rejects the claim if the required time has not passed since policy activation.

Hospital Eligibility Block → Checks whether the hospital type matches the policy requirement for that disease.

Coverage Check Block → Approves the claim fully if under the limit, or partially if treatment cost exceeds coverage.

In [ ]:
POLICY_RULES = {
    "Heart Surgery": {
        "coverage": 300000,
        "waiting_period_years": 2,
        "age_min": 18,
        "age_max": 70,
        "hospital_required": "NABH/NABL Certified",
        "notes": "Only 1 major heart surgery claim allowed in 5 years."
    },
    "Diabetes": {
        "coverage": 50000,
        "waiting_period_years": 2,
        "age_min": 0,
        "age_max": 75,
        "hospital_required": "Any registered hospital",
        "notes": "Consumables and lifestyle medicines are not covered."
    },
    "Cancer Chemo": {
        "coverage": 500000,
        "waiting_period_days": 30,
        "age_min": 0,
        "age_max": 80,
        "hospital_required": "Cancer treatment empanelled hospital",
        "notes": "Chemotherapy sessions only; diagnostics excluded."
    }
}

def parse_date(date_str):
    try:
        return datetime.strptime(date_str, "%Y-%m-%d")
    except:
        return None


def evaluate_claim(name, age, policy_no, disease, hospital_name, hospital_type,
                   treatment_cost, policy_start, claim_date):

    # -------------------------------
    # Basic validation
    # -------------------------------
    if not name or not policy_no or disease == "" or treatment_cost <= 0:
        return "❌ Missing required information."

    if disease not in POLICY_RULES:
        return "❌ This disease is not covered under ICICI Lombard Health+."

    rule = POLICY_RULES[disease]

    # -------------------------------
    # Age Eligibility Check
    # -------------------------------
    if not (rule["age_min"] <= age <= rule["age_max"]):
        return f"❌ Claim Rejected: Age {age} not covered for {disease}."

    # -------------------------------
    # Policy Date Validation
    # -------------------------------
    start = parse_date(policy_start)
    claim = parse_date(claim_date)
    if not start or not claim:
        return "❌ Invalid date format. Use YYYY-MM-DD."

    diff = claim - start

    # waiting period check
    if disease == "Cancer Chemo":
        if diff.days < rule["waiting_period_days"]:
            return f"⏳ Waiting Period Not Completed ({rule['waiting_period_days']} days). Claim Rejected."
    else:
        years = relativedelta(claim, start).years
        if years < rule["waiting_period_years"]:
            return f"⏳ Waiting Period Not Completed ({rule['waiting_period_years']} years). Claim Rejected."

    # -------------------------------
    # Hospital Eligibility Check
    # -------------------------------
    if disease == "Heart Surgery" and hospital_type != "NABH/NABL Certified":
        return "❌ Claim Rejected: Heart surgery allowed only in NABH/NABL certified hospitals."

    if disease == "Cancer Chemo" and hospital_type != "Empanelled Cancer Center":
        return "❌ Claim Rejected: Chemotherapy must be done in an empanelled cancer center."

    # -------------------------------
    # Coverage Check
    # -------------------------------
    coverage = rule["coverage"]

    if treatment_cost > coverage:
        approved = coverage
        extra = treatment_cost - coverage
        return f"""

💡 **Partial Claim Approved**

Disease: {disease}
Policy Holder: {name}
Approved Amount: ₹{approved:,}
Claimed Amount: ₹{treatment_cost:,}
Excess Not Covered: ₹{extra:,}

Reason: Policy coverage limit reached. {rule['notes']}
"""
    else:
        return f"""
✅ **Claim Approved Successfully**

Disease: {disease}
Policy Holder: {name}
Approved Amount: ₹{treatment_cost:,}
Coverage Limit: ₹{coverage:,}

Hospital: {hospital_name}
Condition: {rule['notes']}
"""


# Gradio

In [ ]:
with gr.Blocks(title="ICICI Lombard Health+ Claim AI") as app:
    gr.Markdown("## 🏥 ICICI Lombard Health+ — Medical Claim AI Agent")

    name = gr.Textbox(label="Patient Name")
    age = gr.Number(label="Age", value=30)
    policy_no = gr.Textbox(label="Policy Number")
    disease = gr.Dropdown(["Heart Surgery", "Diabetes", "Cancer Chemo"], label="Disease / Treatment")

    hospital_name = gr.Textbox(label="Hospital Name")
    hospital_type = gr.Dropdown(
        ["NABH/NABL Certified", "Any registered hospital", "Empanelled Cancer Center"],
        label="Hospital Type"
    )

    treatment_cost = gr.Number(label="Treatment Cost (₹)", value=100000)

    policy_start = gr.Textbox(label="Policy Start Date (YYYY-MM-DD)")
    claim_date = gr.Textbox(label="Claim Date (YYYY-MM-DD)")

    btn = gr.Button("Evaluate Claim 🚑")
    output = gr.Markdown()

    btn.click(evaluate_claim,
              inputs=[name, age, policy_no, disease, hospital_name, hospital_type,
                      treatment_cost, policy_start, claim_date],
              outputs=output)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://74676c9b5c3d203a7d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Sample policy number : ICICI-HL-9876543210

NOTE

here we taken Mock data from a ICICI lombard why because they are notprovide real data API to